# Silver Layer — Data Cleaning & Imputation
---
**Objective:** To clean up data and handle missing values ​​without deleting any rows

**Input:** `Bronze_Layer/US_Accidents_Bronze.csv`  

**Output:** `Silver_Layer/US_Accidents_Silver.csv`

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('Silver_Layer', exist_ok=True)
print('Ready | Silver_Layer folder created')

Ready | Silver_Layer folder created


## 2. Load Bronze Data

In [2]:
df = pd.read_csv('Bronze_Layer/US_Accidents_Bronze.csv', low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows | {df.shape[1]} columns')
print(f'Total nulls: {df.isnull().sum().sum():,}')

Loaded: 500,000 rows | 46 columns
Total nulls: 829,873


## 3. Fix Datetime Columns
`Start_Time` و `End_Time` — تحويل لـ datetime وملء الناقص

In [3]:
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce', format='mixed')
df['End_Time']   = pd.to_datetime(df['End_Time'],   errors='coerce', format='mixed')

df['End_Time']   = df['End_Time'].fillna(df['Start_Time'])
df['Start_Time'] = df['Start_Time'].fillna(df['End_Time'])
df['Start_Time'] = df['Start_Time'].fillna(pd.Timestamp('2016-01-01'))
df['End_Time']   = df['End_Time'].fillna(df['Start_Time'])

print(f'Start_Time nulls: {df["Start_Time"].isna().sum()}')
print(f'End_Time nulls  : {df["End_Time"].isna().sum()}')

Start_Time nulls: 0
End_Time nulls  : 0


## 4. Fix Coordinates — End_Lat / End_Lng (44% missing)
المنطق: لو مفيش إحداثيات نهاية → نملأ بإحداثيات البداية

In [4]:
df['End_Lat']   = df['End_Lat'].fillna(df['Start_Lat'])
df['End_Lng']   = df['End_Lng'].fillna(df['Start_Lng'])
df['Start_Lat'] = df['Start_Lat'].fillna(df['End_Lat'])
df['Start_Lng'] = df['Start_Lng'].fillna(df['End_Lng'])

print(f'End_Lat nulls: {df["End_Lat"].isna().sum()}')
print(f'End_Lng nulls: {df["End_Lng"].isna().sum()}')

End_Lat nulls: 0
End_Lng nulls: 0


## 5. Fix Precipitation (28.5% missing)
المنطق: لو Weather_Condition بيقول مطر/تلج → median، غير كده → 0.0

In [5]:
rain_mask   = df['Weather_Condition'].str.contains(
    'Rain|Drizzle|Thunder|Snow|Sleet|Hail|Shower', na=False)
rain_median = df.loc[rain_mask, 'Precipitation(in)'].median()
fill_precip = pd.Series(np.where(rain_mask, rain_median, 0.0), index=df.index)
df['Precipitation(in)'] = df['Precipitation(in)'].fillna(fill_precip)

print(f'Precipitation nulls: {df["Precipitation(in)"].isna().sum()}')
print(f'Rain median used   : {rain_median:.3f} in')

Precipitation nulls: 0
Rain median used   : 0.010 in


## 6. Fix Wind Chill (25.8% missing)
المنطق: حساب معادلة NWS الرسمية لو T<=50F و W>=3mph، غير كده = Temperature

In [6]:
null_wc    = df['Wind_Chill(F)'].isna()
cold_windy = (df['Temperature(F)'] <= 50) & (df['Wind_Speed(mph)'] >= 3)
T          = df['Temperature(F)']
W          = df['Wind_Speed(mph)'].clip(lower=0.1)
wc_formula = 35.74 + 0.6215*T - 35.75*(W**0.16) + 0.4275*T*(W**0.16)

df['Wind_Chill(F)'] = np.where(
    null_wc & cold_windy, wc_formula,
    np.where(null_wc, T, df['Wind_Chill(F)']))

wc_null = df['Wind_Chill(F)'].isna()
if wc_null.any():
    state_wc = df.groupby('State')['Wind_Chill(F)'].transform('median')
    df.loc[wc_null, 'Wind_Chill(F)'] = state_wc[wc_null]
df['Wind_Chill(F)'] = df['Wind_Chill(F)'].fillna(df['Wind_Chill(F)'].median())

print(f'Wind_Chill(F) nulls: {df["Wind_Chill(F)"].isna().sum()}')

Wind_Chill(F) nulls: 0


## 7. Fix Weather Numerics (2-7% missing)
`Wind_Speed / Visibility / Humidity / Temperature / Pressure`  
المنطق: Grouped median على State — أدق من Global median

In [7]:
weather_num_cols = ['Wind_Speed(mph)', 'Visibility(mi)',
                    'Humidity(%)', 'Temperature(F)', 'Pressure(in)']

state_medians  = df.groupby('State')[weather_num_cols].median()
global_medians = df[weather_num_cols].median()

for col in weather_num_cols:
    null_mask = df[col].isna()
    if not null_mask.any(): continue
    df.loc[null_mask, col] = df['State'].map(state_medians[col])[null_mask]
    df[col] = df[col].fillna(global_medians[col])
    print(f'  {col}: {df[col].isna().sum()} nulls remaining')

  Wind_Speed(mph): 0 nulls remaining
  Visibility(mi): 0 nulls remaining
  Humidity(%): 0 nulls remaining
  Temperature(F): 0 nulls remaining
  Pressure(in): 0 nulls remaining


## 8. Fix Categorical Weather (2-2.5% missing)
`Wind_Direction / Weather_Condition`  
المنطق: Mode لنفس الـ State

In [8]:
cat_weather = ['Wind_Direction', 'Weather_Condition']
state_modes = df.groupby('State')[cat_weather].agg(
    lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')

for col in cat_weather:
    null_mask = df[col].isna()
    if not null_mask.any(): continue
    df.loc[null_mask, col] = df['State'].map(state_modes[col])[null_mask]
    df[col] = df[col].fillna('Unknown')
    print(f'  {col}: {df[col].isna().sum()} nulls remaining')

  Wind_Direction: 0 nulls remaining
  Weather_Condition: 0 nulls remaining


## 9. Fix Twilight Columns (0.3% missing)
`Sunrise_Sunset / Civil_Twilight / Nautical_Twilight / Astronomical_Twilight`

In [9]:
twilight_cols = ['Sunrise_Sunset', 'Civil_Twilight',
                 'Nautical_Twilight', 'Astronomical_Twilight']

ss_modes     = df.groupby('Sunrise_Sunset')[twilight_cols].agg(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan)
global_modes = df[twilight_cols].agg(lambda x: x.mode()[0])

for col in twilight_cols:
    null_mask = df[col].isna()
    if not null_mask.any(): continue
    df.loc[null_mask, col] = df['Sunrise_Sunset'].map(ss_modes[col])[null_mask]
    df[col] = df[col].fillna(global_modes[col])
    print(f'  {col}: {df[col].isna().sum()} nulls remaining')

  Sunrise_Sunset: 0 nulls remaining
  Civil_Twilight: 0 nulls remaining
  Nautical_Twilight: 0 nulls remaining
  Astronomical_Twilight: 0 nulls remaining


## 10. Fix Location Columns
`Airport_Code / Street / Timezone / Zipcode / City`

In [10]:
# Airport_Code
null_mask = df['Airport_Code'].isna()
if null_mask.any():
    m = df.groupby(['State','City'])['Airport_Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')
    v = df.set_index(['State','City']).index.map(m)
    df.loc[null_mask,'Airport_Code'] = pd.Series(v, index=df.index)[null_mask]
df['Airport_Code'] = df['Airport_Code'].fillna('Unknown')
print(f'  Airport_Code: {df["Airport_Code"].isna().sum()} nulls')

# Street
df['Street'] = df['Street'].fillna('Unknown')
print(f'  Street: {df["Street"].isna().sum()} nulls')

# Timezone
tz_map = {
    'CA':'US/Pacific','OR':'US/Pacific','WA':'US/Pacific','NV':'US/Pacific',
    'AZ':'US/Mountain','MT':'US/Mountain','ID':'US/Mountain','WY':'US/Mountain',
    'CO':'US/Mountain','NM':'US/Mountain','UT':'US/Mountain',
    'TX':'US/Central','OK':'US/Central','KS':'US/Central','NE':'US/Central',
    'SD':'US/Central','ND':'US/Central','MN':'US/Central','IA':'US/Central',
    'MO':'US/Central','WI':'US/Central','IL':'US/Central','MI':'US/Central',
    'IN':'US/Central','AR':'US/Central','LA':'US/Central','MS':'US/Central',
    'AL':'US/Central','TN':'US/Central',
    'FL':'US/Eastern','GA':'US/Eastern','SC':'US/Eastern','NC':'US/Eastern',
    'VA':'US/Eastern','WV':'US/Eastern','MD':'US/Eastern','DE':'US/Eastern',
    'PA':'US/Eastern','NJ':'US/Eastern','NY':'US/Eastern','CT':'US/Eastern',
    'RI':'US/Eastern','MA':'US/Eastern','NH':'US/Eastern','VT':'US/Eastern',
    'ME':'US/Eastern','KY':'US/Eastern','OH':'US/Eastern','DC':'US/Eastern',
    'HI':'US/Hawaii','AK':'US/Alaska'
}
null_mask = df['Timezone'].isna()
df.loc[null_mask,'Timezone'] = df.loc[null_mask,'State'].map(tz_map)
df['Timezone'] = df['Timezone'].fillna('Unknown')
print(f'  Timezone: {df["Timezone"].isna().sum()} nulls')

# Zipcode
null_mask = df['Zipcode'].isna()
if null_mask.any():
    m = df.groupby(['State','City'])['Zipcode'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')
    v = df.set_index(['State','City']).index.map(m)
    df.loc[null_mask,'Zipcode'] = pd.Series(v, index=df.index)[null_mask]
df['Zipcode'] = df['Zipcode'].fillna('Unknown')
print(f'  Zipcode: {df["Zipcode"].isna().sum()} nulls')

  Airport_Code: 0 nulls
  Street: 0 nulls
  Timezone: 0 nulls
  Zipcode: 0 nulls


## 11. Fix Remaining Columns
`Weather_Timestamp / Description / City / Wind_Chill_Effect`

In [11]:
df['Weather_Timestamp'] = df['Weather_Timestamp'].fillna(df['Start_Time'])
df['Description']       = df['Description'].fillna('No description available')

null_mask = df['City'].isna()
if null_mask.any():
    m = df.groupby(['State','Zipcode'])['City'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')
    v = df.set_index(['State','Zipcode']).index.map(m)
    df.loc[null_mask,'City'] = pd.Series(v, index=df.index)[null_mask]
df['City'] = df['City'].fillna('Unknown')

if 'Wind_Chill_Effect' in df.columns:
    df['Wind_Chill_Effect'] = df['Wind_Chill_Effect'].fillna(df['Temperature(F)'])

print(f'  Weather_Timestamp: {df["Weather_Timestamp"].isna().sum()} nulls')
print(f'  Description      : {df["Description"].isna().sum()} nulls')
print(f'  City             : {df["City"].isna().sum()} nulls')

  Weather_Timestamp: 0 nulls
  Description      : 0 nulls
  City             : 0 nulls


## 12. Validation — Zero Nulls Check

In [12]:
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]

if remaining.empty:
    print('Zero missing values remaining!')
else:
    print('Still missing:')
    print(remaining.to_string())

print(f'Shape: {df.shape[0]:,} rows | {df.shape[1]} columns')

Zero missing values remaining!
Shape: 500,000 rows | 46 columns


## 13. Save Silver Layer

In [13]:
df.to_csv('Silver_Layer/US_Accidents_Silver.csv', index=False)
print(f'Saved: Silver_Layer/US_Accidents_Silver.csv ({df.shape[0]:,} rows)')

Saved: Silver_Layer/US_Accidents_Silver.csv (500,000 rows)
